
# Portfolio Risk Analysis

This notebook evaluates the risk characteristics and benchmark-relative performance of the portfolio.

We focus on:

- Portfolio volatility and covariance
- Correlation and diversification
- Value at Risk (VaR)
- Conditional VaR (CVaR / Expected Shortfall)
- VaR backtesting
- Risk contribution
- Historical stress testing
- Maximum drawdown
- Benchmark comparison
- Portfolio beta

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
adj_close = pd.read_csv(
    "../data/adj_close.csv",
    index_col=0,
    parse_dates=True
)

returns = adj_close.pct_change().dropna()

returns.head()

In [ ]:
weights = np.repeat(1 / len(returns.columns), len(returns.columns))

portfolio_returns = pd.Series(
    np.dot(returns, weights),
    index=returns.index
)

portfolio_returns.head()

# 1. Diversification Analysis

# 1.1- Correlation matrix

In [ ]:

correlation_matrix = returns.corr()
correlation_matrix

# 1.2 - Covariance Matrix

In [ ]:
cov_matrix = returns.cov()
cov_matrix

# 2. Portfolio Risk

# 2.1 - Portfolio Volatility

In [ ]:


portfolio_volatility = np.sqrt(
    np.dot(weights, np.dot(cov_matrix, weights))
)

print(f"Portfolio Volatility: {portfolio_volatility:.2%}")

In [ ]:
annual_portfolio_volatility = portfolio_volatility * np.sqrt(252)

print(f"Annualized Portfolio Volatility: {annual_portfolio_volatility:.2%}")

# 2.2 VAR

In [ ]:

var_95 = portfolio_returns.quantile(0.05)

print(f"95% Daily VaR: {var_95:.2%}")

**Method:** Historical VaR based on the portfolio's observed historical returns.

**Limitation:** Historical VaR assumes that past return patterns are useful for estimating future losses. This assumption may fail during extreme market events.


#Value at Risk (VaR)

VaR estimates the potential loss of a portfolio over a given time period at a chosen confidence level.

The 95% daily VaR printed above is the 5th percentile of the portfolio's historical daily returns.

It means that historically about 95% of trading days had returns better than this value, while the worst 5% of days had larger losses.

VaR focuses on the left tail of the return distribution and helps quantify downside risk.

# 2.3- Conditional VaR / Expected Shortfall

In [ ]:
worst_5_percent = portfolio_returns[portfolio_returns <= var_95] #a dataset containing the worst 5% of historical daily returns.
worst_5_percent 

In [ ]:
cvar_95 = worst_5_percent.mean()

print(f"95% CVaR: {cvar_95:.2%}")

#CVaR / Expected Shortfall

CVaR measures the average loss in the worst 5% of historical trading days.

The 95% CVaR printed above is therefore always a larger loss than the 95% VaR.

CVaR complements VaR because it tells us how severe losses are beyond the VaR threshold.

# 2.4- Individual Stock Volatility

In [ ]:
#Individual stock volatility measures the historical variability of each stock's daily returns.
#Higher volatility means the stock's returns have fluctuated more and therefore represent greater standalone risk.
stock_volatility = returns.std()
stock_volatility

# 2.5- Risk contribution

Risk contribution identifies how much each holding contributes to total portfolio risk.

The simplified measure uses:

Risk Contribution ≈ Weight × Stock Volatility

A more rigorous decomposition accounts for the covariance between assets.

Marginal Contribution to Risk (MCR) measures each asset's incremental contribution to portfolio variance.

Component Contribution to Risk (CCtR) converts this into each asset's share of total portfolio volatility.

The final risk contribution percentages sum to approximately 100%.

In [ ]:
risk_contribution = weights * stock_volatility

risk_contribution

In [ ]:
risk_contribution_pct = risk_contribution * 100
risk_contribution_pct

In [ ]:
## marginal contribution
import numpy as np
marginal_contribution = np.dot(cov_matrix, weights) / portfolio_volatility
marginal_contribution

In [ ]:
## Component Contribution to Risk

cctr = weights * marginal_contribution 

cctr

In [ ]:
cctr_pct = cctr / portfolio_volatility * 100

cctr_pct

In [ ]:
#array to stock names and put the biggest risk contributors at the top.
#who contributes the most risk
risk_share = pd.Series(cctr_pct, index=returns.columns)

risk_share.sort_values(ascending=False)

# 3. Historical Stress Testing

This section evaluates how the portfolio behaved during a historical market stress event.

We use the COVID-19 market shock as a stress scenario and examine:

- Portfolio return during the stress period
- Maximum drawdown during the stress period
- Changes in correlation during market stress

## 3.1 - Select COVID period

In [ ]:
covid_returns = returns.loc["2020-02-01":"2020-04-30"]

covid_returns.head()

## 3.2 - Calculate portfolio returns

In [ ]:
covid_portfolio_returns = np.dot(covid_returns, weights)

covid_portfolio_returns[:5] 


## 3.3 - Calculate cumulative growth

In [ ]:
covid_cumulative = (1 + covid_portfolio_returns).cumprod()
covid_cumulative

## 3.4 - Calculate total COVID-period return

In [ ]:
covid_total_return = covid_cumulative[-1] - 1

print(f"COVID Period Return: {covid_total_return:.2%}")

## 3.5 - Calculate running peak

In [ ]:
covid_peak = np.maximum.accumulate(covid_cumulative)
covid_peak

## 3.6 - Covid drawdown

In [ ]:
covid_drawdown = (covid_cumulative - covid_peak) / covid_peak
covid_drawdown 

In [ ]:
covid_max_drawdown = covid_drawdown.min()

print(f"COVID Maximum Drawdown: {covid_max_drawdown:.2%}")

example:
Peak → €1.00
        ↓
Trough → €0.70

Drawdown = -30%

In [ ]:
trough = np.argmin(covid_drawdown)

print(f"Worst Drawdown Date: {covid_returns.index[trough].date()}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(covid_drawdown * 100)

plt.title("Portfolio Drawdown During COVID-19")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.grid(True)

plt.show()

## 3.7 - Correlation under stress vs. normal conditions.

To assess whether diversification changed during the COVID-19 stress period, we compare the correlation matrix during normal market conditions (2019) with the correlation matrix during the COVID-19 stress period.

A positive value indicates that the correlation between two assets increased during the stress period, while a negative value indicates that it decreased.

In [ ]:
# Calculate daily returns
returns = adj_close.pct_change().dropna()

# Correlation during normal conditions
normal_returns = returns.loc["2019-01-01":"2019-12-31"]
normal_corr = normal_returns.corr()

# Correlation during COVID stress
covid_corr = covid_returns.corr()

# Change in correlation
corr_diff = covid_corr - normal_corr

corr_diff

### Interpretation

The correlation difference matrix shows how the relationships between assets changed during the COVID-19 stress period compared with normal market conditions in 2019.

Positive values mean that two stocks moved more closely together during the stress period. If most values are positive, the assets became more correlated during the crisis.

Higher correlations reduce diversification benefits, because assets become more likely to fall at the same time. The average pairwise correlation below summarises this change in one number for each period.

In [ ]:
# Average pairwise correlation: the mean of the values above the diagonal
def average_correlation(corr):
    upper_triangle = corr.values[np.triu_indices(len(corr), k=1)]
    return upper_triangle.mean()

print(f"Average pairwise correlation, 2019:     {average_correlation(normal_corr):.2f}")
print(f"Average pairwise correlation, COVID-19: {average_correlation(covid_corr):.2f}")

## 4. Risk Analysis Summary

The key risk measures calculated in this notebook are summarised below. The largest risk contributor is taken from the covariance-based Component Contribution to Risk (section 2.5), which accounts for correlations between holdings.

In [ ]:
print(f"Annualized portfolio volatility: {annual_portfolio_volatility:.2%}")
print(f"95% daily VaR:                   {var_95:.2%}")
print(f"95% daily CVaR:                  {cvar_95:.2%}")
print(f"Largest risk contributor:        {risk_share.idxmax()} ({risk_share.max():.1f}% of portfolio risk, weight {weights[0]:.1%})")
print(f"COVID-19 maximum drawdown:       {covid_max_drawdown:.2%}")

Portfolio risk cannot be assessed using volatility alone. Tail risk (VaR and CVaR), drawdowns, individual risk contributions, correlations and stress-period behaviour together give a more complete view of the portfolio's risk profile.

# 5.  Benchmark Analysis — EURO STOXX 50 ETF (net return proxy, EUR)

The benchmark is the Xtrackers EURO STOXX 50 UCITS ETF 1C (`XESC.DE`), an accumulating fund in EUR that tracks the EURO STOXX 50 Net Return index. Because it is a fund, its return is also reduced by its 0.09% annual fee and can differ slightly from the index (tracking difference), so it is a proxy for the index rather than the index itself. All nine portfolio holdings are members of this index. The prices were downloaded and aligned with the portfolio dates in Notebook 1.

In [ ]:
# Benchmark prices in EUR, prepared in Notebook 1
benchmark_data = pd.read_csv(
    "../data/benchmark.csv",
    index_col=0,
    parse_dates=True
)

benchmark_data.head()

In [ ]:
benchmark_prices = benchmark_data.iloc[:, 0]

# Label used in tables and charts
benchmark_name = "EURO STOXX 50 ETF (net return proxy, EUR)"

benchmark_prices.head()

In [ ]:
benchmark_returns = benchmark_prices.pct_change().dropna()

benchmark_returns.head()

In [ ]:
comparison = pd.concat(
    [portfolio_returns, benchmark_returns],
    axis=1,
    join="inner"
)

comparison.columns = ["Portfolio", benchmark_name]

comparison.head()

# 5.1 - BETA Calculation

In [ ]:
beta = (
    comparison["Portfolio"].cov(comparison[benchmark_name])
    / comparison[benchmark_name].var()
)

print(f"Portfolio Beta: {beta:.2f}")

In [ ]:
print("Portfolio-Benchmark Correlation:", comparison["Portfolio"].corr(comparison[benchmark_name]))
print("Portfolio Volatility:", comparison["Portfolio"].std())
print("Benchmark Volatility:", comparison[benchmark_name].std())
print("Portfolio Beta:", beta)

# 5.2 - Benchmark cumulative growth

In [ ]:
benchmark_growth = (1 + benchmark_returns).cumprod()

benchmark_growth.head()

# 5.3 - portfolio_growth

In [ ]:
portfolio_growth = (1 + portfolio_returns).cumprod()

portfolio_growth.head()

# Normalize both

In [ ]:
## Benchmark-Adjusted Performance
portfolio_growth_normalized = portfolio_growth / portfolio_growth.iloc[0]
benchmark_growth_normalized = benchmark_growth / benchmark_growth.iloc[0]

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(portfolio_growth_normalized, label="Portfolio")
plt.plot(benchmark_growth_normalized, label=benchmark_name)

plt.title("Portfolio vs Benchmark – Normalized Growth")
plt.xlabel("Date")
plt.ylabel("Growth of €1")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 5.4 - benchmark performance

In [ ]:
portfolio_growth_normalized.iloc[-1], benchmark_growth_normalized.iloc[-1]

In [ ]:
# Difference in final growth of €1: portfolio minus benchmark
portfolio_growth_normalized.iloc[-1] - benchmark_growth_normalized.iloc[-1]

## 6. VaR Backtesting

VaR backtesting evaluates whether the historical VaR estimate provides a reasonable
representation of the portfolio's observed downside risk.

For a 95% VaR, approximately 5% of historical daily returns are expected to fall
below the VaR threshold.

We compare the observed VaR exceptions with this expected frequency.

In [ ]:
# 95% VaR threshold
var_95 = portfolio_returns.quantile(0.05)

# Identify days when the actual return was worse than the VaR threshold
var_exceptions = portfolio_returns < var_95

# Count the number of exceptions
exception_count = var_exceptions.sum()

# Calculate the observed exception rate
exception_rate = var_exceptions.mean()

print(f"95% VaR: {var_95:.2%}")
print(f"VaR Exceptions: {exception_count}")
print(f"Observed Exception Rate: {exception_rate:.2%}")
print("Expected Exception Rate: 5.00%")

#### VaR Backtesting — Interpretation

The backtest counts the days on which the portfolio's return was worse than the 95% historical VaR and compares this exception rate with the expected rate of 5%.

This is a basic in-sample historical validation. Because the VaR threshold is estimated from the same historical observations used in the backtest, the exception rate will be close to 5% by construction. The result should therefore not be interpreted as evidence that the model will remain accurate under future market conditions.